## Automated RAG evaluation data generation

In [8]:
# !pip install opik llama-index llama-index-agent-openai llama-index-llms-openai --upgrade --quiet

In [9]:
# !pip install -q langchain langchain-community langchain-text-splitters ragas  langchain-ollama

In [10]:
import ragas 
print(ragas.__version__)


0.4.3


In [ ]:
import os
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator

# ── API Key ──
os.environ["GROQ_API_KEY"] = ""
os.environ["OPENROUTER_API_KEY"] = ""

OPENROUTER_KEY = os.environ["OPENROUTER_API_KEY"]

pd.set_option("display.max_colwidth", None)

# ── Load Documents ──
loader = DirectoryLoader(
    "./data/paul_graham",
    glob="**/*.*",
    loader_cls=TextLoader,
    loader_kwargs={"autodetect_encoding": True},
)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=20)
documents = loader.load_and_split(text_splitter)
# The `filename` attribute in metadata is used to identify chunks belonging to the same document. 
# For instance, pages belonging to the same research publication can be identified using the filename.
for document in documents:
    document.metadata['filename'] = document.metadata['source']

print(f"Loaded {len(documents)} chunks")

# ── Raw LLM & Embeddings ──
raw_llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_KEY,
    temperature=0,
)

raw_emb = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    openai_api_key=OPENROUTER_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
)

# ── Quick Test ──
print(raw_llm.invoke("Say hello").content)
r2 = raw_emb.embed_query("What is the second letter of Greek alphabet")
print(f"Embedding dimension: {len(r2)}")

# ── Wrap for Ragas ──
generator_llm = LangchainLLMWrapper(raw_llm)
embeddings = LangchainEmbeddingsWrapper(raw_emb)

# ── Generate Testset ──
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=embeddings,
)

testset = generator.generate_with_langchain_docs(
    documents,
    testset_size=5,
)

test_df = testset.to_pandas()
print(test_df.head())

Loaded 101 chunks
Hello! How can I assist you today?
Embedding dimension: 1536


Applying CustomNodeFilter:   2%|▏         | 2/101 [00:01<01:38,  1.01it/s]Node e2de2e22-beff-49c1-94d0-27771520a662 does not have a summary. Skipping filtering.
Node 8617b3e2-3d34-4b19-be16-71e9f5ce85e9 does not have a summary. Skipping filtering.
Node 4e570f19-8e6a-44ee-aa93-f92fd0117266 does not have a summary. Skipping filtering.
Applying NERExtractor:  33%|███▎      | 33/101 [01:01<01:33,  1.37s/it]Task exception was never retrieved
future: <Task finished name='Task-971' coro=<as_completed.<locals>.sema_coro() done, defined at /Users/mohanreddypanga/opt/anaconda3/envs/agentic_AI/lib/python3.11/site-packages/ragas/async_utils.py:75> exception=AttributeError("'ChatOpenAI' object has no attribute 'aembed_documents'")>
Traceback (most recent call last):
  File "/Users/mohanreddypanga/opt/anaconda3/envs/agentic_AI/lib/python3.11/asyncio/tasks.py", line 277, in __step
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/Users/mohanreddypanga/opt/anaconda3/envs/agentic_AI/li

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


Generating Samples: 100%|██████████| 6/6 [00:10<00:00,  1.81s/it]


                                                                                                                user_input  \
0                            Who was Rich Draves and what role did he play in the early programming experiences described?   
1  Can you describe the process of using Fortran in its early versions, particularly how programs were input and executed?   
2             How does intrinsic motivation relate to the public response to essays, especially when faced with criticism?   
3                              How does the batch model of funding startups contribute to building a community among them?   
4                                                              HN was stress for me, but how did it affect my work on Bel?   

                                                                                                                                                                                                                                                   

In [7]:
test_df.head()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,Who was Rich Draves and what role did he play in the early programming experiences described?,"[What I Worked On\n\nFebruary 2021\n\nBefore college the two main things I worked on, outside of school, were writing and programming. I didn't write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.\n\nThe first programs I tried writing were on the IBM 1401 that our school district used for what was then called ""data processing."" This was in 9th grade, so I was 13 or 14. The school district's 1401 happened to be in the basement of our junior high school, and my friend Rich Draves and I got permission to use it. It was like a mini Bond villain's lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.]","Rich Draves was a friend of the author who, along with the author, got permission to use the IBM 1401 that their school district used for data processing. They explored programming together in the basement of their junior high school, which was described as having an array of alien-looking machines.",Historian of Computing,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
1,"Can you describe the process of using Fortran in its early versions, particularly how programs were input and executed?","[The language we used was an early version of Fortran. You had to type programs on punch cards, then stack them in the card reader and press a button to load the program into memory and run it. The result would ordinarily be to print something on the spectacularly loud printer.]","In the early versions of Fortran, the process of using the language involved typing programs on punch cards. Once the programs were prepared, they were stacked in the card reader. To execute the program, one would press a button to load it into memory and run it. The output of the program would typically be printed on a remarkably loud printer.",Historian of Computing,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
2,"How does intrinsic motivation relate to the public response to essays, especially when faced with criticism?","[<1-hop>\n\n[10] This was the first instance of what is now a familiar experience, and so was what happened next, when I read the comments and found they were full of angry people. How could I claim that Lisp was better than other languages? Weren't they all Turing complete? People who see the responses to essays I write sometimes tell me how sorry they feel for me, but I'm not exaggerating when I reply that it has always been like this, since the very beginning. It comes with the territory. An essay must tell readers things they don't already know, and some people dislike being told such things.\n\n[11] People put plenty of stuff on the internet in the 90s of course, but putting something online is not the same as publishing it online. Publishing online means you treat the online version as the (or at least a) primary version., <2-hop>\n\nIt's not that unprestigious types of work are good per se. But when you find yourself drawn to some kind of work despite its current lack of prestige, it's a sign both that there's something real to be discovered there, and that you have the right kind of motives. Impure motives are a big danger for the ambitious. If anything is going to lead you astray, it will be the desire to impress people. So while working on things that aren't prestigious doesn't guarantee you're on the right track, it at least guarantees you're not on the most common type of wrong one.]","Intrinsic motivation is crucial when dealing with public responses to essays, as it helps writers navigate criticism. The context highlights that negative 